<a href="https://colab.research.google.com/github/ZyrenKing/no-music/blob/main/no_music_full_lastet_VVV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U demucs
!pip install diffq

In [ ]:
#@title 🚀 Razi's Ultra-Fast High-Quality Isolation (Updated) 🚀

import subprocess
import sys
import torch
import shutil
import os
import gc
from google.colab import drive

# --- الإعدادات ---
input_mp3 = "test.mp3" #@param {type:"string"}
# htdemucs هو أفضل نموذج يجمع بين السرعة والجودة العالية
model = "htdemucs"

def process_audio():
    # 1. تنظيف الذاكرة والتحقق من الـ GPU
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        device = "cuda"
        print("✅ مفعّل: تم اكتشاف GPU. ستكون المعالجة سريعة جداً.")
    else:
        device = "cpu"
        print("⚠️ تنبيه: لم يتم اكتشاف GPU. المعالجة ستكون بطيئة جداً.")
        print("⚠️ يرجى الذهاب إلى Runtime -> Change runtime type واختيار T4 GPU.")

    if not os.path.exists('/content/drive'):
        print("🔗 جاري ربط Google Drive...")
        drive.mount('/content/drive')

    # 2. إعداد الأمر (استخدام htdemucs مع تحسينات السرعة)
    # --two-stems=vocals لعزل الموسيقى عن الصوت
    # -j 4 للاستفادة من تعدد الأنوية
    command = f'python -m demucs -n {model} -d {device} --mp3 --two-stems=vocals -j 4 "{input_mp3}"'

    print(f"\n⚙️ جارٍ المعالجة (النموذج: {model})...")

    try:
        process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

        for char in iter(lambda: process.stdout.read(1), ''):
            sys.stdout.write(char)
            sys.stdout.flush()

        process.wait()

        if process.returncode == 0:
            print("\n\n✅ تم العزل بنجاح!")

            base_name = os.path.splitext(input_mp3)[0]
            # مسار الملف الناتج في htdemucs
            source_file = f"/content/separated/{model}/{base_name}/vocals.mp3"
            destination = f"/content/drive/MyDrive/{base_name}_vocal_high_quality.mp3"

            if os.path.exists(source_file):
                print("⏳ جاري حفظ النسخة في Google Drive...")
                shutil.copy(source_file, destination)
                print(f"✅ اكتمل الحفظ: {base_name}_vocal_high_quality.mp3")
            else:
                print("❌ حدثت مشكلة في العثور على الملف الناتج.")

            gc.collect()
        else:
            print(f"\n\n❌ خطأ في المعالجة (Code: {process.returncode})")

    except Exception as e:
        print(f"\n\n❌ خطأ تقني: {e}")

if __name__ == "__main__":
    if not os.path.exists(input_mp3):
        print(f"❌ خطأ: الملف '{input_mp3}' غير موجود. تأكد من رفعه إلى المجلد الرئيسي أولاً.")
    else:
        process_audio()